In [1]:
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import balanced_accuracy_score
import warnings

warnings.filterwarnings('ignore')

file_path = '../../../data/data_for_final_models/AgglomerativeClustering_generated_features.csv'
data = pd.read_csv(file_path)

X = data.drop(columns='Cluster')
y = data['Cluster']

kf = KFold(n_splits=5, shuffle=True, random_state=42)

catboost_model = CatBoostClassifier(random_state=42, verbose=False)
lr_model = LogisticRegression(random_state=42)
rf_model = RandomForestClassifier(random_state=42)
et_model = ExtraTreesClassifier(random_state=42)
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)

voting_preds = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    voting_clf = VotingClassifier(
        estimators=[
            ('catboost', catboost_model),
            ('lr', lr_model),
            ('rf', rf_model),
            ('et', et_model),
            ('gb', gb_model)
        ],
        voting='soft'
    )
    
    voting_clf.fit(X_train, y_train)
    
    voting_preds.append(voting_clf.predict(X_val))

voting_preds = [item for sublist in voting_preds for item in sublist]

accuracy = balanced_accuracy_score(y, voting_preds)
print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.3410
